### **Initializing**
Environemnts and External Libraries

---

*   Google Query & Colab utilites
*   Common Data-Sci: pandas, matplot etc.

In [ ]:
import os
%load_ext google.colab.data_table
from google.colab.data_table import DataTable
from google.colab import auth
from google.cloud.bigquery import magics
from google.cloud import bigquery
import scipy
import matplotlib.pyplot as plt
import pandas as pd

!pip install amsterdamumcdb
import amsterdamumcdb as adb

#Changing Default plot-styles
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams["figure.dpi"] = 144

DataTable.max_rows = 30000
DataTable.max_columns = 50


PROJECT_ID = "Your ID Here"
DATASET_PROJECT_ID = 'amsterdamumcdb'
DATASET_ID = 'version1_5_0'
LOCATION = 'eu'


config_gbq = {'query':
          {'defaultDataset': {
              "datasetId": DATASET_ID,
              "projectId": DATASET_PROJECT_ID
              },
           'Location': LOCATION}
           }


os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
auth.authenticate_user()

Default_Conf = bigquery.job.QueryJobConfig(default_dataset=DATASET_PROJECT_ID + "." + DATASET_ID)
magics.context.default_query_job_config = Default_Conf

Data_Query = bigquery.job.QueryJobConfig()
Request = bigquery.Client(project=PROJECT_ID,location=LOCATION, default_query_job_config=Default_Conf)


**Querying AmsterdamUMCdb**

In [ ]:
Data_set = Request.query('''WITH Patient_Gender AS (
    SELECT
        o.person_id,
        CASE
            WHEN o.value_as_concept_id = 8507 THEN 'Male'
            WHEN o.value_as_concept_id = 8532 THEN 'Female'
            ELSE 'Unknown'
        END AS gender
    FROM
        observation o
    WHERE
        o.observation_concept_id = 4083588 -- Patient sex concept ID
        AND o.value_as_concept_id IN (8532, 8507) -- Female and Male
        AND o.provider_id IS NOT NULL -- Satisfy partition elimination requirement
),

Patient_Weight AS (
    SELECT
        person_id,
        value_as_number AS weight
    FROM
        measurement
    WHERE
        provider_id IS NULL
        AND measurement_concept_id = 3013762 -- Body weight Measured
),

LOS_Data AS (
    SELECT
        v.person_id,
        v.visit_occurrence_id,
        SUM(TIMESTAMP_DIFF(visit_end_datetime, visit_start_datetime, HOUR)) / 24.0 AS los_days,
        EXTRACT(YEAR FROM visit_start_datetime) - year_of_birth AS age
    FROM
        visit_occurrence v
    INNER JOIN
        person o ON v.person_id = o.person_id
    LEFT JOIN
        concept c ON o.gender_concept_id = c.concept_id
    GROUP BY
        v.person_id, v.visit_occurrence_id, age

),

Ranked_Measurements AS (
    SELECT
        m.person_id,
        m.measurement_datetime,
        m.measurement_source_value,
        CASE WHEN specimen_concept_id = 4047496 THEN 'arterial' END AS specimen,
        c2.concept_name,
        m.value_as_number,
        ROW_NUMBER() OVER (PARTITION BY c2.concept_id ORDER BY m.measurement_datetime) AS rn
    FROM
        measurement m
    INNER JOIN
        concept c2 ON m.measurement_concept_id = c2.concept_id
    INNER JOIN
        specimen s ON m.person_id = s.person_id
        AND m.measurement_datetime = s.specimen_datetime
        AND s.specimen_concept_id = 4047496
    WHERE
        NOT m.provider_id IS NULL -- ignore unvalidated device data
        AND m.measurement_type_concept_id = 32856 -- Lab
        AND c2.concept_id IN (
            3010421, -- pH of Blood
            3044904, -- Oxygen content in Blood
            3006576, -- Bicarbonate [Moles/volume] in blood
            3000285, -- Sodium [Moles/volume] in Blood
            3005456, -- Potassium [Moles/volume] in Blood
            3047181, -- Lactate [Moles/volume] in Blood
            3018572, -- Chloride [Moles/volume] in Blood
            3021119  -- Calcium.ionized [Moles/volume] in Blood
        )
),

Death_Data AS (
    SELECT
        person_id,
        1 AS deceased,
        death_date
    FROM
        death

)

SELECT
    pg.person_id,
    pg.gender,
    pw.weight,
    ld.visit_occurrence_id,
    ld.los_days,
    ld.age,
    rm.measurement_datetime,
    rm.measurement_source_value,
    rm.specimen,
    rm.concept_name,
    rm.value_as_number,
    dd.death_date,
    dd.deceased
FROM
    Patient_Gender pg
INNER JOIN
    Patient_Weight pw ON pg.person_id = pw.person_id
INNER JOIN
    LOS_Data ld ON pg.person_id = ld.person_id
INNER JOIN
    Ranked_Measurements rm ON pg.person_id = rm.person_id
LEFT JOIN
    Death_Data dd ON pg.person_id = dd.person_id

''').to_dataframe()


In [ ]:
Data_set.info()
Data_set.shape[1]
Data_set.describe()
print(Data_set['person_id'].nunique())